CIFAR 10 なんでこのデータか意味わからんから、AG_news

GRU, RNN, BiRNN, LSTM

In [ ]:
#%pip install -q datasets
from datasets import load_dataset

ds = load_dataset("ag_news")
ds["train"][0]


Note: you may need to restart the kernel to use updated packages.


README.md: 0.00B [00:00, ?B/s]

c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nutzer\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py:90: UserWarning: A Nu

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [2]:
# 必要なら（既に入っていれば不要）
# %pip install -q datasets torch

import re
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset


# ----------------------------
# データ読み込み（Hugging Face datasets）
# ----------------------------
ds = load_dataset("ag_news")  # ds["train"], ds["test"] が使える

class AGNewsHFDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # HFのag_newsは label が 0..3（torchtextと違って 1..4 じゃない）
        return int(item["label"]), item["text"]


train_ds = AGNewsHFDataset(ds["train"])
test_ds  = AGNewsHFDataset(ds["test"])


# ----------------------------
# tokenizer（torchtextの basic_english っぽい簡易版）
# ----------------------------
def basic_english_tokenizer(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # 記号を空白に
    text = re.sub(r"\s+", " ", text).strip()   # 連続空白を詰める
    return text.split()


# ----------------------------
# vocab（torchtextなしで自作）
# ----------------------------
class Vocab:
    def __init__(self, stoi, itos, default_index: int):
        self.stoi = stoi
        self.itos = itos
        self.default_index = default_index

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token: str):
        return self.stoi.get(token, self.default_index)

    def __call__(self, tokens):
        return [self[token] for token in tokens]


def build_vocab_from_texts(text_iter, tokenizer, specials=("<unk>", "<pad>"), min_freq=2):
    counter = Counter()
    for text in text_iter:
        toks = tokenizer(text)
        if not toks:
            toks = ["<unk>"]
        counter.update(toks)

    itos = list(specials)
    # 安定するように freq desc → token asc で並べる
    for tok, freq in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        if freq >= min_freq and tok not in specials:
            itos.append(tok)

    stoi = {tok: i for i, tok in enumerate(itos)}
    default_index = stoi["<unk>"]
    return Vocab(stoi=stoi, itos=itos, default_index=default_index)


# train から vocab 作成
vocab = build_vocab_from_texts(
    (ds["train"][i]["text"] for i in range(len(ds["train"]))),
    tokenizer=basic_english_tokenizer,
    specials=("<unk>", "<pad>"),
    min_freq=2
)
pad_idx = vocab["<pad>"]


def text_pipeline(x: str):
    tokens = basic_english_tokenizer(x)
    if not tokens:
        tokens = ["<unk>"]
    return vocab(tokens)

def label_pipeline(y: int):
    # HFのag_newsは 0..3 のままでOK
    return int(y)


# ----------------------------
# DataLoader 用 collate（padding + lengths）
# ----------------------------
def collate_batch(batch):
    labels = []
    sequences = []
    lengths = []

    for (label, text) in batch:
        labels.append(label_pipeline(label))
        ids = torch.tensor(text_pipeline(text), dtype=torch.long)
        sequences.append(ids)
        lengths.append(ids.size(0))

    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    tokens_padded = pad_sequence(sequences, batch_first=True, padding_value=pad_idx)
    return tokens_padded, lengths, labels


batch_size = 64
trainloader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_batch, num_workers=0)
testloader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_batch, num_workers=0)



c:\Users\Nutzer\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py:90: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.2)
  return _bootstrap._gcd_import(name[level:], package, level)


## モデル

In [6]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

# --------
# GRU
# --------
class NetGRU(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        last_hidden = h_n[-1]                 # (N, H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


# --------
# RNN（単方向）
# --------
class NetRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers=1, batch_first=True, nonlinearity="tanh")
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        last_hidden = h_n[-1]                 # (N, H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


# --------
# BiRNN（双方向RNN）
# --------
class NetBiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(
            embed_dim, hidden_size, num_layers=1, batch_first=True,
            bidirectional=True, nonlinearity="tanh"
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # 双方向なので 2H

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed)
        # h_n: (layers*dirs, N, H) = (2, N, H) なので最後2つが forward/backward
        forward_last = h_n[-2]                # (N, H)
        backward_last = h_n[-1]               # (N, H)
        last_hidden = torch.cat([forward_last, backward_last], dim=1)  # (N, 2H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


# --------
# LSTM（単方向）
# --------
class NetLSTM(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        x = self.embedding(tokens)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, c_n) = self.lstm(packed)
        last_hidden = h_n[-1]                 # (N, H)
        logits = self.fc(last_hidden)         # (N, C)
        return logits


In [ ]:
# BiRNN（双方向RNN）で AG_NEWS（文章分類：4クラス）を学習する “完全版” サンプル
# 目的：CIFAR10みたいに「Dataset → DataLoader → Net → 学習 → 評価」をRNNで再現

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from torchtext.datasets import AG_NEWS
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


# ----------------------------
# モデル定義（BiRNN）
# __init__：層の構成
# forward：データの流れ
# ----------------------------
class BiRNN(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_layers=1, num_classes=4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # bidirectional=True -> 方向が2つ（forward/backward）
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        # 本の双方向出力の合成（前向き + 後ろ向き）に合わせて hidden_size -> num_classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, tokens, lengths):
        """
        tokens:  (N, T)  パディング済みの単語ID
        lengths: (N,)    各文の長さ（パディング前）
        return:  (N, num_classes) logits
        """
        # ①長さでソート（pack用）※enforce_sorted=Falseでも動くが、教科書式に明示
        lengths_sorted, perm_idx = lengths.sort(0, descending=True)
        tokens_sorted = tokens[perm_idx]

        # ②Embedding
        x = self.embedding(tokens_sorted)  # (N, T, embed_dim)

        # ③可変長をpackしてRNNへ（パッド部分を無視して計算）
        packed = pack_padded_sequence(x, lengths_sorted.cpu(), batch_first=True)

        # ④RNN
        # h_n: (num_layers * num_directions, N, hidden_size)
        _, h_n = self.rnn(packed)

        # ⑤ソートを元に戻す（h_n はソート順のままなので戻す）
        _, unperm_idx = perm_idx.sort(0)
        h_n = h_n[:, unperm_idx, :]

        # ⑥最終層の forward/backward の隠れ状態を合成して分類
        # num_layers>=1 のとき、最後の層の forward は -2、backward は -1 に入る
        h_fwd = h_n[-2]  # (N, hidden_size)
        h_bwd = h_n[-1]  # (N, hidden_size)
        h = h_fwd + h_bwd  # (N, hidden_size)  ※concatにするなら fcの入力次元も変える

        logits = self.fc(h)  # (N, num_classes)
        return logits


# ----------------------------
# 前処理：tokenize -> vocab -> ID化
# ----------------------------
def yield_tokens(data_list, tokenizer):
    for label, text in data_list:
        yield tokenizer(text)


def main(epochs=2, lr=0.05, momentum=0.9, batch_size=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # tokenizer
    tokenizer = get_tokenizer("basic_english")

    # Dataset（CIFAR10のdownload=True的に、自動DLされる）
    train_list = list(AG_NEWS(split="train"))
    test_list = list(AG_NEWS(split="test"))

    # vocab
    special_tokens = ["<unk>", "<pad>"]
    vocab = build_vocab_from_iterator(
        yield_tokens(train_list, tokenizer),
        specials=special_tokens,
        min_freq=2,
    )
    vocab.set_default_index(vocab["<unk>"])
    pad_idx = vocab["<pad>"]

    # pipeline
    def text_pipeline(text: str):
        return vocab(tokenizer(text))

    def label_pipeline(label: int):
        return int(label) - 1  # AG_NEWSは 1..4 なので 0..3 へ

    # DataLoader用 collate（padding + lengths）
    def collate_batch(batch):
        labels = []
        seqs = []
        lengths = []

        for (label, text) in batch:
            labels.append(label_pipeline(label))
            ids = torch.tensor(text_pipeline(text), dtype=torch.long)
            seqs.append(ids)
            lengths.append(ids.size(0))

        labels = torch.tensor(labels, dtype=torch.long)
        lengths = torch.tensor(lengths, dtype=torch.long)

        tokens_padded = pad_sequence(seqs, batch_first=True, padding_value=pad_idx)  # (N, T)
        return tokens_padded, lengths, labels

    trainloader = DataLoader(train_list, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
    testloader = DataLoader(test_list, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

    # Model / Loss / Optim
    net = BiRNN(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr, momentum=momentum)

    # ----------------------------
    # 学習
    # ----------------------------
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 0):
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)     # (N, 4)
            loss = criterion(outputs, labels)  # labels: (N,)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 199:
                print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}")
                running_loss = 0.0

    print("Finished Training")

    # ----------------------------
    # 評価
    # ----------------------------
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            outputs = net(tokens, lengths)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {100.0 * correct / total:.2f}%")
    return net


if __name__ == "__main__":
    main()


In [ ]:
# GRU
main(NetGRU)

In [ ]:

# RNN
main(NetRNN)

In [ ]:
# BiRNN
main(NetBiRNN)

In [ ]:





# LSTM
main(NetLSTM)


[1,   200] loss: 1.337
[1,   400] loss: 1.118
[1,   600] loss: 0.800
[1,   800] loss: 0.625
[1,  1000] loss: 0.536
[1,  1200] loss: 0.490
[1,  1400] loss: 0.463
[1,  1600] loss: 0.441
[1,  1800] loss: 0.435
[2,   200] loss: 0.379
[2,   400] loss: 0.381
[2,   600] loss: 0.376
[2,   800] loss: 0.374
[2,  1000] loss: 0.362
[2,  1200] loss: 0.365
[2,  1400] loss: 0.360
[2,  1600] loss: 0.353
[2,  1800] loss: 0.351
Finished Training
Test Accuracy: 87.51%
[1,   200] loss: 1.369
[1,   400] loss: 1.345
[1,   600] loss: 1.346
[1,   800] loss: 1.350
[1,  1000] loss: 1.398
[1,  1200] loss: 1.476
[1,  1400] loss: 1.590
[1,  1600] loss: 1.656
[1,  1800] loss: 1.615
[2,   200] loss: 1.589
[2,   400] loss: 1.673
[2,   600] loss: 1.658
[2,   800] loss: 1.728
[2,  1000] loss: 1.653
[2,  1200] loss: 1.691
[2,  1400] loss: 1.682
[2,  1600] loss: 1.573
[2,  1800] loss: 1.592
Finished Training
Test Accuracy: 25.92%
[1,   200] loss: 1.354
[1,   400] loss: 1.319
[1,   600] loss: 1.284
[1,   800] loss: 1.284


NetLSTM(
  (embedding): Embedding(44118, 64, padding_idx=1)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)

白本に載っていたBiRNNのコード

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


# =========================================================
# 191-192p: 双方向RNN（BiRNN）  ※穴埋め反映
#   あ = hidden_size*2
#   う = unsqueeze(-1)
# =========================================================
class BiRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.RNN = nn.RNN(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True
        )  # 双方向RNN

        # (あ) = hidden_size*2
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # 全結合層

    def forward(self, x, seq_lengths, masks):
        """
        x: (N, T, input_size)   ※埋め込み済みを想定
        seq_lengths: (N,)
        masks: (N, T)  1=有効, 0=pad
        """

        # 入力のシーケンスの長さで並び替え
        seq_lengths, perm_idx = seq_lengths.sort(0, descending=True)
        x = x[perm_idx]
        masks = masks[perm_idx]

        # パディング（pack）
        x = pack_padded_sequence(x, seq_lengths.cpu(), batch_first=True, enforce_sorted=True)

        # 初期隠れ状態（双方向なので *2）
        h0 = x.data.new_zeros(self.num_layers * 2, masks.size(0), self.hidden_size)

        # RNNセルを通して各時刻のシーケンスを処理
        out, _ = self.RNN(x, h0)

        # unpack
        out, _ = pad_packed_sequence(out, batch_first=True)  # (N, T, 2H)

        # 並び替えを元に戻す
        _, unperm_idx = perm_idx.sort(0)
        out = out[unperm_idx]
        masks = masks[unperm_idx]

        # pad部分を無効化（う = unsqueeze(-1)）
        out = out * masks.unsqueeze(-1)  # (N, T, 2H)

        # --- もし「前向き+後ろ向きを足す」実装にしたいなら ---
        # out = out[:, :, :self.hidden_size] + out[:, :, self.hidden_size:]  # (N, T, H)
        # self.fc = nn.Linear(self.hidden_size, num_classes)  # ← fc を H 入力に変更する

        out = self.fc(out)  # (N, T, num_classes)
        return out  # 出力を返す


# =========================================================
# 192p: BiRNNLoss  ※穴埋め反映
#   え = loss * masks
#   お = masks
# =========================================================
class BiRNNLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # masks を掛けたいので reduction='none' が実用上必須（誌面だと省略されがち）
        self.loss_fn = nn.CrossEntropyLoss(reduction="none")

    def forward(self, outputs, targets, masks):
        """
        outputs: (N, T, C)
        targets: (N, T)
        masks:   (N, T)
        """
        targets = targets.view(-1)
        outputs = outputs.view(-1, outputs.size(-1))
        masks = masks.view(-1).float()

        loss = self.loss_fn(outputs, targets)        # (N*T,)
        loss = torch.sum(loss * masks) / torch.sum(masks)  # (え)/(お)
        return loss


# =========================================================
# あなたの「学習+評価（tokens,lengths,labels）」に繋ぐラッパー
#  - DataLoaderは (tokens, lengths, labels) のままでOK
#  - masks は tokens から自動生成する
# =========================================================
class NetBiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, pad_idx, embed_dim=64, hidden_size=128, num_classes=4, num_layers=1):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.birnn = BiRNN(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            num_classes=num_classes
        )

    def forward(self, tokens, lengths):
        # tokens: (N, T)
        x = self.embedding(tokens)  # (N, T, embed_dim)
        masks = (tokens != self.pad_idx).float()  # (N, T)

        # BiRNNは (N, T, C) を返す
        out = self.birnn(x, lengths, masks)

        # 文章分類用：最後の有効stepの出力だけ取り出す -> (N, C)
        idx = (lengths - 1).clamp(min=0)  # (N,)
        last_logits = out[torch.arange(out.size(0), device=out.device), idx]
        return last_logits


# =========================================================
# 学習 + 評価（あなたの形をほぼ維持：Netだけ差し替え）
# =========================================================
def main(NetClass):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    net = NetClass(vocab_size=len(vocab), pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=0.05, momentum=0.9)

    for epoch in range(2):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 0):
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)  # (N, C)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 200 == 199:
                print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}")
                running_loss = 0.0

    print("Finished Training")

    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            outputs = net(tokens, lengths)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {100.0 * correct / total:.2f}%")


# 使い方：BiRNNで回す
# main(NetBiRNNClassifier)


In [5]:
main(NetBiRNNClassifier)

[1,   200] loss: 1.393
[1,   400] loss: 1.373
[1,   600] loss: 1.376
[1,   800] loss: 1.362
[1,  1000] loss: 1.329
[1,  1200] loss: 1.285
[1,  1400] loss: 1.259
[1,  1600] loss: 1.255
[1,  1800] loss: 1.195
[2,   200] loss: 1.165
[2,   400] loss: 1.141
[2,   600] loss: 1.165
[2,   800] loss: 1.173
[2,  1000] loss: 1.269
[2,  1200] loss: 1.208
[2,  1400] loss: 1.314
[2,  1600] loss: 1.364
[2,  1800] loss: 1.416
Finished Training
Test Accuracy: 30.80%
